In [0]:
from pyspark.sql import functions as F

CATALOG = dbutils.widgets.get("catalog")

SILVER_WEATHER = f"{CATALOG}.skybound_silver.weather_reports_silver"
SILVER_AIRPORTS = f"{CATALOG}.skybound_silver.airports_enriched_silver"
GOLD = f"{CATALOG}.skybound_gold"

CHECKPOINT_PATH = f"/Volumes/{CATALOG}/skybound_bronze/checkpoints/weather_to_gold"

In [0]:
from pyspark.sql import functions as F

airports = (
    spark.read
    .table(SILVER_AIRPORTS)
    .filter(F.col("country_name") != "Russia")
    .select(
        F.col("ident").alias("airport_icao"),
        F.col("airport_name"),
        F.col("country_name").alias("airport_country"),
        F.col("country_code"),
        F.col("latitude_deg").alias("airport_latitude"),
        F.col("longitude_deg").alias("airport_longitude"),
        F.col("elevation_ft").alias("airport_elevation"),
    )
)

(
    spark.readStream
    .table(SILVER_WEATHER)
    .withWatermark("obs_time", "2 hours")

    .join(airports, F.col("station_id") == F.col("airport_icao"), "inner")

    .withColumn("obs_date", F.to_date("obs_time"))
    .withColumn("date_hour", F.date_format("obs_time", "yyyyMMddHH").cast("long"))

    .select(
        "station_id",
        "airport_name",
        "airport_country",
        "country_code",
        "airport_latitude",
        "airport_longitude",
        "airport_elevation",

        "date_hour",
        "obs_time",
        "obs_date",
        "temp_c",           "dewpoint_c",
        "wind_speed_kt",    "wind_gust_kt",
        "wind_dir_degrees", "wind_dir_variable",
        "visibility_sm",    "visibility_unlimited",
        "altim_hpa",
        "cover",            "cloud_base_ft",
        "cover_description",
        "wx_string",
        F.col("wx_string_decoded").alias("weather_description"),
        "has_precipitation", "has_freezing",
        "has_thunderstorm",  "has_fog",
        "flight_category",   "flight_category_name",
        "category_severity", "is_ifr",
        "metar_type",        "raw_ob",
        "_ingestion_timestamp",
    )

    .writeStream
    .trigger(availableNow=True)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable(f"{GOLD}.fact_airport_weather_reports")
)